In [ ]:
import pandas as pd
import pycountry
import json
from os import listdir as ls
from datetime import datetime, timedelta
from matplotlib import pyplot as plt 

from emu_renewal.inputs import DATA_PATH, get_apple_mobility, get_worldbank_national_pop, \
    get_country_vacc_data, get_indicator_series_from_who_data, get_country_vacc_data, get_undesa_national_pop
from emu_renewal.run import find_run_start_time, find_run_end_time

In [ ]:
def get_mob_countries():
    pop_threshold = 1e6
    gmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
    fbmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
    amob_avail = []
    amob_unavail = []
    for c in pycountry.countries:
        try:
            iso3 = c.alpha_3
            get_apple_mobility(iso3)
            amob_avail.append(iso3)
        except:
            amob_unavail.append(iso3)
    return list(set(amob_avail + fbmob_avail + gmob_avail))

# json.dump(get_mob_countries(), open(DATA_PATH / "config/mob_countries.json", "w"))

In [ ]:
any_mob_avail = json.load(open(DATA_PATH / "config/mob_countries.json", "r"))

In [ ]:
cases_data = {}
deaths_data = {}
pops = {}
for c in any_mob_avail:
    try:
        pops[c] = get_worldbank_national_pop(c)
    except:
        pops[c] = get_undesa_national_pop(c)
    cases_data[c] = get_indicator_series_from_who_data("New_cases", c)
    deaths_data[c] = get_indicator_series_from_who_data("New_deaths", c)

In [ ]:
no_cases = []
no_deaths = []
neg_cases = []
neg_deaths = []
for c, cases in cases_data.items():
    if cases.size == 0:
        no_cases.append(c)
    elif cases.min() < 0.0:
        neg_cases.append(c)
for c, deaths in deaths_data.items():
    if deaths.size == 0:
        no_deaths.append(c)
    elif deaths.min() < 0.0:
        neg_deaths.append(c)

In [ ]:
mob_inds_avail = [c for c in any_mob_avail if c not in set(no_cases + no_deaths + neg_cases + neg_deaths)]

In [ ]:
def plot_cases_deaths(country, end):

    def pop_normalise(x):
        return x / pops[country]
    def pop_denormalise(x):
        return x * pops[country]
    
    country_name = pycountry.countries.lookup(country).name
    country_cases = cases_data[country]
    country_deaths = deaths_data[country]
    c_cases = country_cases[country_cases.index < end]
    c_deaths = country_deaths[country_deaths.index < end]
    fig, axes = plt.subplots(2, 1, sharex=True)
    axes[0].plot(c_cases.index, c_cases, marker="o", linewidth=0.0, markersize=3.0)
    axes[0].set_title(f"cases, {country_name} ({country})")
    axes[0].secondary_yaxis("right", functions=(pop_normalise, pop_denormalise))
    axes[1].plot(c_deaths.index, c_deaths, marker="o", linewidth=0.0, markersize=3.0)
    axes[1].set_title(f"deaths, {country_name} ({country})")
    axes[1].secondary_yaxis("right", functions=(pop_normalise, pop_denormalise))
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=70)
    fig.tight_layout()
    fig.savefig(f"indicators_{country}.png")

In [ ]:
end_time = datetime(2021, 6, 30)

In [ ]:
def is_change_repeated(data, n_repeats):
    repeat_change = (data.diff().diff().abs() < 1e-10) & (data > 0.0)
    is_repeat = repeat_change.astype(int)
    multirepeat = is_repeat.rolling(window=n_repeats).sum()
    return (multirepeat == float(n_repeats)).any()

In [ ]:
regular_changes = []
for c in mob_inds_avail:
    c_data = cases_data[c]
    d_data = deaths_data[c]
    filt_d_data = d_data[d_data.index < end_time]
    filt_c_data = c_data[c_data.index < end_time]
    if is_change_repeated(filt_d_data, 7) or is_change_repeated(filt_c_data, 7):
        regular_changes.append(c)

In [ ]:
included = [c for c in mob_inds_avail if c not in regular_changes]

In [ ]:
for c in ["DMA"]:
    plot_cases_deaths(c, end_time)